# Exp7.3 — Training Strategy Decomposition

Analysis-only notebook. Primary metric: **final LIF test balanced accuracy** after every training strategy is normalized to the same $\beta=0.5$ LIF spike-count deployment.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / 'notebooks').exists():
    repo = repo.parent
base = repo / 'notebooks' / 'artifacts' / 'experiment_7_3_training_strategy_decomposition' / 'training_strategy_decomposition_v1'
manifest = json.loads((base / 'manifest.json').read_text())
runs = pd.read_csv(base / 'method_runs.csv')
summary = pd.read_csv(base / 'method_summary.csv')
two_stage = pd.read_csv(base / 'two_stage_matrix_summary.csv')
contrasts = pd.read_csv(base / 'contrast_summary.csv')
manifest


## 1. All methods under common final LIF deployment

In [ ]:
method_order = manifest['e2e_methods'] + manifest['two_stage_methods']
view = summary.set_index('method').loc[method_order].reset_index()
cols = [
    'method', 'group', 'backbone_objective', 'w_objective', 'w_train_readout',
    'analog_test_ba_mean', 'final_lif_test_ba_mean', 'final_lif_test_ba_std',
    'analog_to_lif_gap_pp_mean', 'l2_wholecount_probe_test_ba_mean',
    'l2_fixed250_probe_test_ba_mean'
]
view[cols]


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(view))
ax.bar(x, view['final_lif_test_ba_mean'])
ax.errorbar(x, view['final_lif_test_ba_mean'], yerr=view['final_lif_test_ba_std'], fmt='none', capsize=3)
ax.set_xticks(x, view['method'], rotation=65, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp7.3: all methods normalized to final LIF readout')
ax.set_ylim(0, 1)
fig.tight_layout()


## 2. Two-stage factorial matrix

Rows isolate Stage-1 backbone objective; columns isolate Stage-2 W objective and W-training readout.

In [ ]:
matrix = two_stage.pivot_table(
    index='backbone_objective',
    columns=['w_train_readout', 'w_objective'],
    values='final_lif_test_ba_mean',
)
matrix


## 3. Representation quality vs final LIF accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(view['l2_wholecount_probe_test_ba_mean'], view['final_lif_test_ba_mean'])
for _, row in view.iterrows():
    ax.annotate(row['method'].split('_')[0], (row['l2_wholecount_probe_test_ba_mean'], row['final_lif_test_ba_mean']))
ax.set_xlabel('Frozen-L2 WholeCount linear-probe test BA')
ax.set_ylabel('Final LIF test BA')
ax.set_title('Representation accessibility vs deployed LIF classification')
fig.tight_layout()


## 4. Pre-registered key contrasts

Positive delta means the left method has higher final-LIF test BA.

In [ ]:
contrasts[['contrast', 'left_method', 'right_method', 'final_lif_delta_pp_mean', 'final_lif_delta_pp_std']]
